# 2.1 Data raw ingestion

(1) Lectura del Excel original como texto, (2) contrato de esquema y renombrado a `snake_case`, (3) tipado explícito sin limpiar valores,
(4) metadatos de trazabilidad por fila, (5) validaciones y persistencia.

Artefactos persistidos: una tabla Parquet por hoja. En esta capa **no se corrige nada**: solo tipos y metadatos.

| Artefacto | Ruta |
|---|---|
| Entrada | `data/0_source/KPIS_historico.xlsx` |
| Salida KPIs | `data/1_raw/kpis_raw.parquet` |
| Salida catálogo indicadores | `data/1_raw/catalogo_indicadores_raw.parquet` |
| Salida catálogo entornos | `data/1_raw/catalogo_entornos_raw.parquet` |


In [1]:
from pathlib import Path
import hashlib

import pandas as pd

SOURCE_PATH = Path("../../data/0_source/KPIS_historico.xlsx")
RAW_ROOT = Path("../../data/1_raw")
RAW_ROOT.mkdir(parents=True, exist_ok=True)

KPIS_RAW_PATH = RAW_ROOT / "kpis_raw.parquet"
CAT_IND_RAW_PATH = RAW_ROOT / "catalogo_indicadores_raw.parquet"
CAT_ENT_RAW_PATH = RAW_ROOT / "catalogo_entornos_raw.parquet"

# Contrato: columnas esperadas en el origen -> nombre estándar y tipo en raw
CONTRATOS = {
    "query": {
        "salida": KPIS_RAW_PATH,
        "columnas": {
            "Frente": ("frente", "string"),
            "Corte": ("corte", "Int64"),
            "Codigo_EQU": ("codigo_equipo", "string"),
            "EQU": ("nombre_equipo", "string"),
            "Indicador": ("indicador", "string"),
            "Resultado": ("resultado", "Float64"),
            "Meta": ("meta", "Float64"),
            "Cumplimiento": ("cumplimiento_reportado", "Float64"),
        },
    },
    "catalogo_indicadores": {
        "salida": CAT_IND_RAW_PATH,
        "columnas": {
            "Frente": ("frente", "string"),
            "Indicador": ("indicador", "string"),
            "Definición (qué mide y para qué)": ("definicion", "string"),
            "Unidad": ("unidad", "string"),
        },
    },
    "catalogo_entornos": {
        "salida": CAT_ENT_RAW_PATH,
        "columnas": {
            "Codigo_EQU": ("codigo_equipo", "string"),
            "Tipo": ("tipo_equipo", "string"),
            "EQU": ("nombre_equipo", "string"),
            "Codigo_Padre": ("codigo_padre", "string"),
            "Nombre_Padre": ("nombre_padre", "string"),
        },
    },
}


## 1. Lectura y contrato de esquema


In [2]:
def hash_archivo(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def leer_hoja(hoja, contrato):
    df = pd.read_excel(SOURCE_PATH, sheet_name=hoja, dtype=str)
    esperadas = list(contrato["columnas"])
    faltantes = set(esperadas) - set(df.columns)
    sobrantes = set(df.columns) - set(esperadas)
    if faltantes or sobrantes:
        raise ValueError(f"{hoja}: contrato roto | faltantes={faltantes} | sobrantes={sobrantes}")
    return df[esperadas]


HASH_ORIGEN = hash_archivo(SOURCE_PATH)
origen = {hoja: leer_hoja(hoja, c) for hoja, c in CONTRATOS.items()}

for hoja, df in origen.items():
    print(hoja, "| filas:", len(df), "| contrato OK")
print("sha256 origen:", HASH_ORIGEN[:16], "...")


query | filas: 15188 | contrato OK
catalogo_indicadores | filas: 13 | contrato OK
catalogo_entornos | filas: 64 | contrato OK
sha256 origen: 1e6120364ab60547 ...


## 2. Tipado explícito y metadatos de ingesta


In [3]:
def tipar(df, contrato, hoja):
    out = pd.DataFrame(index=df.index)
    for col_origen, (col_destino, tipo) in contrato["columnas"].items():
        texto = df[col_origen].astype("string").str.strip()
        if tipo in ("Int64", "Float64"):
            valores = pd.to_numeric(texto, errors="coerce")
            perdidos = (texto.notna() & valores.isna()).sum()
            assert perdidos == 0, f"{hoja}.{col_origen}: {perdidos} valores no convertibles"
            out[col_destino] = valores.astype(tipo)
        else:
            out[col_destino] = texto
    # Fila del Excel (encabezado = fila 1) para rastrear cualquier registro hasta el origen
    out["_fila_origen"] = (df.index + 2).astype("int64")
    out["_hoja"] = hoja
    out["_archivo_origen"] = SOURCE_PATH.name
    out["_hash_archivo"] = HASH_ORIGEN
    return out


raw = {hoja: tipar(df, CONTRATOS[hoja], hoja) for hoja, df in origen.items()}

for hoja, df in raw.items():
    print(hoja, "| tipos:", dict(df.dtypes.astype(str).drop(["_hoja", "_archivo_origen", "_hash_archivo"])))


query | tipos: {'frente': 'string', 'corte': 'Int64', 'codigo_equipo': 'string', 'nombre_equipo': 'string', 'indicador': 'string', 'resultado': 'Float64', 'meta': 'Float64', 'cumplimiento_reportado': 'Float64', '_fila_origen': 'int64'}
catalogo_indicadores | tipos: {'frente': 'string', 'indicador': 'string', 'definicion': 'string', 'unidad': 'string', '_fila_origen': 'int64'}
catalogo_entornos | tipos: {'codigo_equipo': 'string', 'tipo_equipo': 'string', 'nombre_equipo': 'string', 'codigo_padre': 'string', 'nombre_padre': 'string', '_fila_origen': 'int64'}


## 3. Validaciones y persistencia


In [4]:
for hoja, df in raw.items():
    assert len(df) == len(origen[hoja]), f"{hoja}: cambió el número de filas"
    assert df["_fila_origen"].is_unique
    nulos_origen = origen[hoja].isna().sum().sum()
    nulos_raw = df[[c for c, _ in CONTRATOS[hoja]["columnas"].values()]].isna().sum().sum()
    assert nulos_origen == nulos_raw, f"{hoja}: la ingesta alteró nulos ({nulos_origen} vs {nulos_raw})"

for hoja, df in raw.items():
    salida = CONTRATOS[hoja]["salida"]
    df.to_parquet(salida, index=False)
    print(hoja, "| filas:", len(df), "| guardado:", salida)


query | filas: 15188 | guardado: ..\..\data\1_raw\kpis_raw.parquet
catalogo_indicadores | filas: 13 | guardado: ..\..\data\1_raw\catalogo_indicadores_raw.parquet
catalogo_entornos | filas: 64 | guardado: ..\..\data\1_raw\catalogo_entornos_raw.parquet
